# Tổng hợp các mô hình từ đầu đến hiện tại để so sánh các độ đo và đảm bảo tính thực tiễn của các mô hình áp dụng

## **Phương pháp Deep Learning, Supervised Learning áp dụng trong bài toán TicTacToe**

## Môi trường game TTT

In [1]:
import numpy as np
import random
from copy import deepcopy
class action_space:
    def __init__(self, n):
        self.n = n
    
class observation_space:
    def __init__(self, n):
        self.shape = (n,)
class ttt:
    def __init__(self): 
        self.action_space = action_space(9)
        self.observation_space = observation_space(9)
        self.info = ""         
        self.cellcenter = {1:(-200,-200), 2:(0,-200), 3:(200,-200),
                           4:(-200,0),    5:(0,0),    6:(200,0),
                           7:(-200,200),  8:(0,200),  9:(200,200)} 
        self.reset()
        
    def sample(self):
        return random.choice(self.validinputs)   
    def reset(self):  
        self.turn = "X"
        self.rounds = 1
        self.validinputs = list(range(1, 10))
        self.occupied = {"X": [], "O": []}
        self.state = np.array([0]*9)
        self.done = False
        self.reward = 0     
        return self.state        
        
    def step(self, inp):
        inp = int(inp)
        self.occupied[self.turn].append(inp)
        self.state[inp - 1] = 1 if self.turn == "X" else -1
        self.validinputs.remove(inp) 
        
        if self.win_game():
            self.done = True
            self.reward = 1 if self.turn == "X" else -1
            self.validinputs = []
        elif self.rounds == 9:
            self.done = True
            self.reward = 0
            self.validinputs = []
        else:
            self.rounds += 1
            self.turn = "O" if self.turn == "X" else "X"             
        return self.state, self.reward, self.done, self.info
                    
    def win_game(self):
        lst = self.occupied[self.turn]
        lines = [
            [1, 2, 3], [4, 5, 6], [7, 8, 9],
            [1, 4, 7], [2, 5, 8], [3, 6, 9],
            [1, 5, 9], [3, 5, 7]
        ]
        for line in lines:
            if line[0] in lst and line[1] in lst and line[2] in lst:
                return True
        return False
print("✅ Đã khởi tạo môi trường ttt() độc lập, sẵn sàng chạy trên Kaggle!")

✅ Đã khởi tạo môi trường ttt() độc lập, sẵn sàng chạy trên Kaggle!


## Định nghĩa các thuật toán cần thiết: Minimax_ab()

In [2]:
# =============================================================================
# THUẬT TOÁN MINIMAX ALPHA-BETA PRUNING (EXPERT PLAYER CHO TICTACTOE)
# =============================================================================
from copy import deepcopy
from random import choice

def maximized_payoff_ttt(env, reward, done, alpha, beta):
    if done:
        return -1 if reward != 0 else 0
    if alpha is None: alpha = -2
    if beta is None: beta = -2
    
    best_payoff = alpha if env.turn == "X" else beta         
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, info = env_copy.step(m)  
        opponent_payoff = maximized_payoff_ttt(env_copy, reward, done, alpha, beta)
        my_payoff = -opponent_payoff 
        if my_payoff > best_payoff:        
            best_payoff = my_payoff
            if env.turn == "X": alpha = best_payoff
            if env.turn == "O": beta = best_payoff 
        if alpha >= -beta:
            break        
    return best_payoff        

def MiniMax_ab(env):
    wins = []
    ties = []
    losses = []  
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, info = env_copy.step(m) 
        if done and reward != 0:
            return m 
        opponent_payoff = maximized_payoff_ttt(env_copy, reward, done, -2, -2)  
        my_payoff = -opponent_payoff 
        if my_payoff == 1:
            wins.append(m)
        elif my_payoff == 0:
            ties.append(m)
        else:
            losses.append(m)
            
    if len(wins) > 0:
        return choice(wins)
    elif len(ties) > 0:
        return choice(ties)
    return env.sample()

print("✅ Đã khởi tạo hàm MiniMax_ab(env) độc lập!")

✅ Đã khởi tạo hàm MiniMax_ab(env) độc lập!


## Định nghĩa lớp tích chập CNN

In [3]:
import numpy as np

board = np.array([[1,0,0],
                   [1,-1,-1],
                   [1,0,0]]).reshape(-1,3,3,1) 

In [4]:
# Create a vertical filter
vertical_filter = np.array([[0,1,0], 
                   [0,1,0],
                   [0,1,0]]).reshape(3,3,1,1)  

In [5]:
import tensorflow as tf

# Ép kiểu dữ liệu sang tf.float32 để tương thích với tf.nn.conv2d
board_float = tf.cast(board, tf.float32)
filter_float = tf.cast(vertical_filter, tf.float32)

# Áp dụng phép tích chập (Convolution 2D)
result = tf.nn.conv2d(board_float, filter_float, strides=1, padding="SAME")

# In kết quả dạng ma trận 3x3
print(result.numpy().reshape(3, 3))

[[ 2. -1. -1.]
 [ 3. -1. -1.]
 [ 2. -1. -1.]]


2026-09-21 13:27:29.041286: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Tạo dữ liệu mẫu

Tạo dữ liệu mẫu bằng cách sử dụng model cây Minimax làm người chơi chuyên nghiệp, người chơi nghiệp dư với 50% đi giống với cây Minimax, 50% đi ngẫu nhiên

In [6]:
import numpy as np

def expert(env):
    return MiniMax_ab(env)    

def non_expert(env):
    if np.random.rand() < 0.5:
        return MiniMax_ab(env)
    else:
        return env.sample()

In [7]:
from copy import deepcopy

env = ttt()

def one_game(episode):
    history = []
    state = env.reset()  
    # Người chơi non-expert đi trước một nửa số ván (các ván chẵn)
    if episode % 2 == 0:
        action = non_expert(env)
        state, reward, done, _ = env.step(action)
    while True:   
        action = expert(env) 
        if episode % 2 == 0:
            statei = deepcopy(-state)
        else:
            statei = deepcopy(state)            
        actioni = deepcopy(action)
        history.append((statei, actioni))
        state, reward, done, _ = env.step(action)
        if done:
            break
        action = non_expert(env)
        state, reward, done, _ = env.step(action)     
        if done:
            break
    return history

# Test thử nghiệm 1 ván
sample_history = one_game(0)
print(f"Mẫu dữ liệu ghi nhận từ 1 ván: {len(sample_history)} nước đi.")
print(sample_history)

Mẫu dữ liệu ghi nhận từ 1 ván: 4 nước đi.
[(array([ 0,  0, -1,  0,  0,  0,  0,  0,  0]), 5), (array([ 0,  0, -1,  0,  1, -1,  0,  0,  0]), 9), (array([-1,  0, -1,  0,  1, -1,  0,  0,  1]), 2), (array([-1,  1, -1,  0,  1, -1,  0, -1,  1]), 4)]


In [8]:
import os
import pickle
import time

# Đường dẫn thư mục làm việc (Tự nhận diện Kaggle hoặc Local)
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
os.makedirs(WORKING_DIR, exist_ok=True)
GAMES_DATA_PATH = os.path.join(WORKING_DIR, "games_ttt.p")

TOTAL_GAMES = 10000
SAVE_INTERVAL = 1000  # Lưu checkpoint định kỳ mỗi 1.000 ván

# 1. Kiểm tra nếu đã có checkpoint từ trước
results = []
start_episode = 0

if os.path.exists(GAMES_DATA_PATH):
    try:
        with open(GAMES_DATA_PATH, "rb") as fp:
            saved_data = pickle.load(fp)
            if isinstance(saved_data, dict) and "results" in saved_data:
                results = saved_data["results"]
                start_episode = saved_data.get("episode", 0)
            else:
                results = saved_data
                start_episode = TOTAL_GAMES
        print(f"🔄 Tìm thấy dữ liệu checkpoint! Đã có {len(results)} mẫu từ {start_episode}/{TOTAL_GAMES} ván.")
    except Exception as e:
        print(f"⚠️ Lỗi đọc file cũ ({e}), bắt đầu mô phỏng mới...")
        results = []
        start_episode = 0

# 2. Chạy mô phỏng tiếp tục từ start_episode
if start_episode < TOTAL_GAMES:
    print(f"🚀 Bắt đầu mô phỏng từ ván {start_episode + 1} đến {TOTAL_GAMES}...")
    t0 = time.time()
    for episode in range(start_episode, TOTAL_GAMES):
        history = one_game(episode)
        results += history
        
        # Lưu checkpoint định kỳ
        if (episode + 1) % SAVE_INTERVAL == 0 or (episode + 1) == TOTAL_GAMES:
            checkpoint_payload = {
                "results": results,
                "episode": episode + 1
            }
            with open(GAMES_DATA_PATH, "wb") as fp:
                pickle.dump(checkpoint_payload, fp)
            elapsed = time.time() - t0
            print(f"💾 Checkpoint: Đã hoàn thành {episode + 1}/{TOTAL_GAMES} ván | Thu thập: {len(results)} mẫu ({elapsed:.1f}s)")
else:
    print(f"✅ Đã đủ {TOTAL_GAMES} ván ({len(results)} mẫu trạng thái cờ). Không cần mô phỏng lại!")

🚀 Bắt đầu mô phỏng từ ván 1 đến 10000...
💾 Checkpoint: Đã hoàn thành 1000/10000 ván | Thu thập: 3835 mẫu (2412.7s)
💾 Checkpoint: Đã hoàn thành 2000/10000 ván | Thu thập: 7672 mẫu (4166.0s)
💾 Checkpoint: Đã hoàn thành 3000/10000 ván | Thu thập: 11465 mẫu (5905.2s)
💾 Checkpoint: Đã hoàn thành 4000/10000 ván | Thu thập: 15283 mẫu (7675.1s)
💾 Checkpoint: Đã hoàn thành 5000/10000 ván | Thu thập: 19141 mẫu (9449.3s)
💾 Checkpoint: Đã hoàn thành 6000/10000 ván | Thu thập: 22937 mẫu (11208.4s)
💾 Checkpoint: Đã hoàn thành 7000/10000 ván | Thu thập: 26739 mẫu (12966.7s)
💾 Checkpoint: Đã hoàn thành 8000/10000 ván | Thu thập: 30540 mẫu (14769.8s)
💾 Checkpoint: Đã hoàn thành 9000/10000 ván | Thu thập: 34356 mẫu (16543.9s)
💾 Checkpoint: Đã hoàn thành 10000/10000 ván | Thu thập: 38172 mẫu (18321.1s)


```python
# simulate the game 1000 times and record all games
results = []        
for episode in range(100):
    history=one_game(episode)
    results+=history 
```


## Huấn luyện 2 model policy

In [9]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten

fast_model = Sequential()
fast_model.add(Conv2D(filters=128, 
    kernel_size=(3,3),padding="same",activation="relu",
                 input_shape=(3,3,1)))
fast_model.add(Flatten())
fast_model.add(Dense(units=64, activation="relu"))
fast_model.add(Dense(units=64, activation="relu"))
fast_model.add(Dense(9, activation='softmax'))
fast_model.compile(loss='categorical_crossentropy',
                   optimizer='adam', 
                   metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
strong_model = Sequential()
strong_model.add(Conv2D(filters=128, 
    kernel_size=(3,3),padding="same",activation="relu",
                 input_shape=(3,3,1)))
strong_model.add(Flatten())
strong_model.add(Dense(units=64, activation="relu"))
strong_model.add(Dense(units=64, activation="relu"))
strong_model.add(Dense(units=64, activation="relu"))
strong_model.add(Dense(9, activation='softmax'))
strong_model.compile(loss='categorical_crossentropy',
                   optimizer='adam', 
                   metrics=['accuracy'])

```python
import pickle
import numpy as np
with open('files/games_ttt.p','rb') as fp:
    games=pickle.load(fp)

states=[]
actions=[]
for x in games:
    state=x[0]
    action=to_categorical(x[1]-1,9)
    states.append(state)
    actions.append(action)

X=np.array(states).reshape((-1, 3, 3, 1))
y=np.array(actions).reshape((-1, 9))
```

In [11]:
import numpy as np
import pickle
import os
from tensorflow.keras.utils import to_categorical

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
GAMES_DATA_PATH = os.path.join(WORKING_DIR, "games_ttt.p")

# Lấy dữ liệu từ RAM hoặc từ file checkpoint
if 'results' in globals() and len(results) > 0:
    games = results
    print(f"✅ Lấy dữ liệu trực tiếp từ biến 'results' trong RAM: {len(games)} mẫu.")
elif os.path.exists(GAMES_DATA_PATH):
    with open(GAMES_DATA_PATH, 'rb') as fp:
        loaded = pickle.load(fp)
        games = loaded["results"] if isinstance(loaded, dict) and "results" in loaded else loaded
    print(f"✅ Tải dữ liệu từ file {GAMES_DATA_PATH}: {len(games)} mẫu.")
else:
    raise FileNotFoundError("Chưa có dữ liệu mẫu. Hãy chạy Cell 14 để mô phỏng dữ liệu!")

states = []
actions = []
for x in games:
    state = x[0]
    action = to_categorical(x[1] - 1, 9)
    states.append(state)
    actions.append(action)

X = np.array(states).reshape((-1, 3, 3, 1))
y = np.array(actions).reshape((-1, 9))

print(f"📊 Kích thước dữ liệu huấn luyện: X = {X.shape}, y = {y.shape}")

✅ Lấy dữ liệu trực tiếp từ biến 'results' trong RAM: 38172 mẫu.
📊 Kích thước dữ liệu huấn luyện: X = (38172, 3, 3, 1), y = (38172, 9)


```python
# Train the fast policy network for 100 epochs
fast_model.fit(X, y, epochs=100, verbose=1)
fast_model.save('files/fast_ttt.h5')
```

In [12]:
import os
import json
import tensorflow as tf
from tensorflow.keras.models import load_model

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
FAST_MODEL_PATH = os.path.join(WORKING_DIR, "fast_ttt.h5")
FAST_STATE_PATH = os.path.join(WORKING_DIR, "fast_checkpoint.json")
TOTAL_EPOCHS = 100

# Callback tự động lưu checkpoint sau mỗi epoch
class EpochCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, model_path, state_path, save_freq=5):
        super().__init__()
        self.model_path = model_path
        self.state_path = state_path
        self.save_freq = save_freq
        
    def on_epoch_end(self, epoch, logs=None):
        current_ep = epoch + 1
        if current_ep % self.save_freq == 0 or current_ep == TOTAL_EPOCHS:
            self.model.save(self.model_path)
            state = {"completed_epoch": current_ep}
            with open(self.state_path, "w") as f:
                json.dump(state, f)
            print(f"\n💾 [Fast Model] Đã lưu checkpoint tại Epoch {current_ep}/{TOTAL_EPOCHS}")

# Kiểm tra checkpoint đã lưu trước đó
initial_epoch = 0
if os.path.exists(FAST_MODEL_PATH) and os.path.exists(FAST_STATE_PATH):
    try:
        with open(FAST_STATE_PATH, "r") as f:
            state = json.load(f)
            initial_epoch = state.get("completed_epoch", 0)
        if initial_epoch > 0:
            fast_model = load_model(FAST_MODEL_PATH)
            print(f"🔄 Đã tải Fast Model checkpoint từ Epoch {initial_epoch}!")
    except Exception as e:
        print(f"Không thể tải checkpoint ({e}), khởi động huấn luyện mới.")
        initial_epoch = 0

if initial_epoch >= TOTAL_EPOCHS:
    print(f"✅ Fast Model đã hoàn tất toàn bộ {TOTAL_EPOCHS} epochs từ trước!")
else:
    print(f"🚀 Bắt đầu huấn luyện Fast Model từ Epoch {initial_epoch + 1} đến {TOTAL_EPOCHS}...")
    checkpoint_cb = EpochCheckpoint(FAST_MODEL_PATH, FAST_STATE_PATH, save_freq=5)
    fast_model.fit(
        X, y, 
        epochs=TOTAL_EPOCHS, 
        initial_epoch=initial_epoch, 
        callbacks=[checkpoint_cb],
        verbose=1
    )
    fast_model.save(FAST_MODEL_PATH)
    print(f"✅ Đã lưu Fast Model hoàn chỉnh tại: {FAST_MODEL_PATH}")

🚀 Bắt đầu huấn luyện Fast Model từ Epoch 1 đến 100...
Epoch 1/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5326 - loss: 1.2203
Epoch 2/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6086 - loss: 0.9339
Epoch 3/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6173 - loss: 0.8908
Epoch 4/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6194 - loss: 0.8719
Epoch 5/100
1181/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6254 - loss: 0.8544


💾 [Fast Model] Đã lưu checkpoint tại Epoch 5/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6228 - loss: 0.8568
Epoch 6/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6244 - loss: 0.8506
Epoch 7/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6258 - loss: 0.8412
Epoch 8/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6268 - loss: 0.8337
Epoch 9/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6252 - loss: 0.8308
Epoch 10/100
1187/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6330 - loss: 0.8184


💾 [Fast Model] Đã lưu checkpoint tại Epoch 10/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6294 - loss: 0.8238
Epoch 11/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6304 - loss: 0.8222
Epoch 12/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6294 - loss: 0.8196
Epoch 13/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6302 - loss: 0.8145
Epoch 14/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6309 - loss: 0.8124
Epoch 15/100
1187/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6364 - loss: 0.7985


💾 [Fast Model] Đã lưu checkpoint tại Epoch 15/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6306 - loss: 0.8099
Epoch 16/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6317 - loss: 0.8072
Epoch 17/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6303 - loss: 0.8050
Epoch 18/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6300 - loss: 0.8034
Epoch 19/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6316 - loss: 0.8015
Epoch 20/100
1187/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6351 - loss: 0.7949


💾 [Fast Model] Đã lưu checkpoint tại Epoch 20/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6334 - loss: 0.7995
Epoch 21/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6319 - loss: 0.7976
Epoch 22/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6310 - loss: 0.7969
Epoch 23/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6321 - loss: 0.7956
Epoch 24/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6326 - loss: 0.7942
Epoch 25/100
1180/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6350 - loss: 0.7863

...


💾 [Fast Model] Đã lưu checkpoint tại Epoch 85/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6413 - loss: 0.7635
Epoch 86/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6427 - loss: 0.7656
Epoch 87/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6415 - loss: 0.7637
Epoch 88/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6422 - loss: 0.7615
Epoch 89/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6414 - loss: 0.7636
Epoch 90/100
1184/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6442 - loss: 0.7648


💾 [Fast Model] Đã lưu checkpoint tại Epoch 90/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6408 - loss: 0.7631
Epoch 91/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6423 - loss: 0.7632
Epoch 92/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6387 - loss: 0.7617
Epoch 93/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6417 - loss: 0.7636
Epoch 94/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6431 - loss: 0.7623
Epoch 95/100
1179/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6451 - loss: 0.7627


💾 [Fast Model] Đã lưu checkpoint tại Epoch 95/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6405 - loss: 0.7663
Epoch 96/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6431 - loss: 0.7622
Epoch 97/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6430 - loss: 0.7612
Epoch 98/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6424 - loss: 0.7645
Epoch 99/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6417 - loss: 0.7643
Epoch 100/100
1190/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6427 - loss: 0.7637


💾 [Fast Model] Đã lưu checkpoint tại Epoch 100/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6415 - loss: 0.7611


✅ Đã lưu Fast Model hoàn chỉnh tại: /kaggle/working/fast_ttt.h5


```python
strong_model.fit(X, y, epochs=100, verbose=1)
strong_model.save('files/strong_ttt.h5')
```

In [13]:
import os
import json
import tensorflow as tf
from tensorflow.keras.models import load_model

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./files"
STRONG_MODEL_PATH = os.path.join(WORKING_DIR, "strong_ttt.h5")
STRONG_STATE_PATH = os.path.join(WORKING_DIR, "strong_checkpoint.json")
TOTAL_EPOCHS = 100

initial_epoch = 0
if os.path.exists(STRONG_MODEL_PATH) and os.path.exists(STRONG_STATE_PATH):
    try:
        with open(STRONG_STATE_PATH, "r") as f:
            state = json.load(f)
            initial_epoch = state.get("completed_epoch", 0)
        if initial_epoch > 0:
            strong_model = load_model(STRONG_MODEL_PATH)
            print(f"🔄 Đã tải Strong Model checkpoint từ Epoch {initial_epoch}!")
    except Exception as e:
        print(f"Không thể tải checkpoint ({e}), khởi động huấn luyện mới.")
        initial_epoch = 0

if initial_epoch >= TOTAL_EPOCHS:
    print(f"✅ Strong Model đã hoàn tất toàn bộ {TOTAL_EPOCHS} epochs từ trước!")
else:
    print(f"🚀 Bắt đầu huấn luyện Strong Model từ Epoch {initial_epoch + 1} đến {TOTAL_EPOCHS}...")
    checkpoint_cb = EpochCheckpoint(STRONG_MODEL_PATH, STRONG_STATE_PATH, save_freq=5)
    strong_model.fit(
        X, y, 
        epochs=TOTAL_EPOCHS, 
        initial_epoch=initial_epoch, 
        callbacks=[checkpoint_cb],
        verbose=1
    )
    strong_model.save(STRONG_MODEL_PATH)
    print(f"✅ Đã lưu Strong Model hoàn chỉnh tại: {STRONG_MODEL_PATH}")

🚀 Bắt đầu huấn luyện Strong Model từ Epoch 1 đến 100...
Epoch 1/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5138 - loss: 1.2708
Epoch 2/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6077 - loss: 0.9372
Epoch 3/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6157 - loss: 0.8918
Epoch 4/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6210 - loss: 0.8685
Epoch 5/100
1189/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6262 - loss: 0.8520


💾 [Fast Model] Đã lưu checkpoint tại Epoch 5/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6249 - loss: 0.8528
Epoch 6/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6253 - loss: 0.8442
Epoch 7/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6273 - loss: 0.8354
Epoch 8/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6255 - loss: 0.8307
Epoch 9/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6282 - loss: 0.8245
Epoch 10/100
1189/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6267 - loss: 0.8237


💾 [Fast Model] Đã lưu checkpoint tại Epoch 10/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6277 - loss: 0.8215
Epoch 11/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6276 - loss: 0.8160
Epoch 12/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6320 - loss: 0.8117
Epoch 13/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6293 - loss: 0.8106
Epoch 14/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6323 - loss: 0.8070
Epoch 15/100
1192/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6385 - loss: 0.7973


💾 [Fast Model] Đã lưu checkpoint tại Epoch 15/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6330 - loss: 0.8031
Epoch 16/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6317 - loss: 0.8023
Epoch 17/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6314 - loss: 0.7996
Epoch 18/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6325 - loss: 0.7970
Epoch 19/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6323 - loss: 0.7966
Epoch 20/100
1189/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6354 - loss: 0.7873


💾 [Fast Model] Đã lưu checkpoint tại Epoch 20/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6338 - loss: 0.7918
Epoch 21/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6310 - loss: 0.7908
Epoch 22/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6314 - loss: 0.7899
Epoch 23/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6347 - loss: 0.7875
Epoch 24/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6323 - loss: 0.7870
Epoch 25/100
1183/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6354 - loss: 0.7873

...


💾 [Fast Model] Đã lưu checkpoint tại Epoch 85/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6422 - loss: 0.7671
Epoch 86/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6405 - loss: 0.7613
Epoch 87/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6412 - loss: 0.7627
Epoch 88/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6410 - loss: 0.7677
Epoch 89/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6409 - loss: 0.7639
Epoch 90/100
1182/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6443 - loss: 0.7610


💾 [Fast Model] Đã lưu checkpoint tại Epoch 90/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6423 - loss: 0.7631
Epoch 91/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6434 - loss: 0.7612
Epoch 92/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6383 - loss: 0.7658
Epoch 93/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6415 - loss: 0.7686
Epoch 94/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6417 - loss: 0.7620
Epoch 95/100
1192/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6463 - loss: 0.7542


💾 [Fast Model] Đã lưu checkpoint tại Epoch 95/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6400 - loss: 0.7658
Epoch 96/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6410 - loss: 0.7669
Epoch 97/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6435 - loss: 0.7603
Epoch 98/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6431 - loss: 0.7608
Epoch 99/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6423 - loss: 0.7674
Epoch 100/100
1185/1193 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6440 - loss: 0.7668


💾 [Fast Model] Đã lưu checkpoint tại Epoch 100/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6431 - loss: 0.7643


✅ Đã lưu Strong Model hoàn chỉnh tại: /kaggle/working/strong_ttt.h5


## Chạy thử nghiệm

**Môi trường util10**

In [14]:
# =============================================================================
# MÔI TRƯỜNG CỐT LÕI MCTS & CÔNG THỨC UCT TÍCH HỢP POLICY NETWORK
# =============================================================================
import random
from copy import deepcopy
from math import sqrt, log
import numpy as np

# 1. Các bước cơ bản của MCTS truyền thống (Chương 8)
def expand(env, move):
    env_copy = deepcopy(env)
    state, reward, done, info = env_copy.step(move)
    return env_copy, done, reward

def simulate(env_copy, done, reward):
    if done:
        return reward
    while True:
        move = env_copy.sample()
        state, reward, done, info = env_copy.step(move)
        if done:
            return reward

def backpropagate(env, move, reward, counts, wins, losses):
    counts[move] = counts.get(move, 0) + 1
    if reward == 1 and env.turn == "X":
        wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "O":
        wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "X":
        losses[move] = losses.get(move, 0) + 1
    elif reward == 1 and env.turn == "O":
        losses[move] = losses.get(move, 0) + 1
    return counts, wins, losses

def select(env, counts, wins, losses, temperature=1.414):
    for k in env.validinputs:
        if counts[k] == 0:
            return k
    N = sum(counts.values())
    scores = {}
    for k in env.validinputs:
        vi = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
        exploration = temperature * sqrt(log(N) / counts[k])
        scores[k] = vi + exploration
    return max(scores, key=scores.get)

def next_best_move(counts, wins, losses):
    scores = {}
    for k in counts.keys():
        if counts[k] == 0:
            scores[k] = -float('inf')
        else:
            scores[k] = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
    return max(scores, key=scores.get)

def mcts(env, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    for _ in range(num_rollouts):
        move = select(env, counts, wins, losses, temperature)
        env_copy, done, reward = expand(env, move)
        reward = simulate(env_copy, done, reward)
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
    return next_best_move(counts, wins, losses)

# 2. Các hàm mở rộng kết hợp Policy Network (Chương 10)
gamma = 10  # Hệ số cân bằng trọng số mạng Policy

def mix_select(env, ps, counts, wins, losses, temperature=1.414):
    for k in env.validinputs:
        if counts[k] == 0:
            return k
    N = sum(counts.values())
    scores = {}
    for k in env.validinputs:
        weighted_pi = gamma * ps[k] / (1 + counts[k])
        vi = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
        exploration = temperature * sqrt(log(N) / counts[k])
        scores[k] = vi + exploration + weighted_pi
    return max(scores, key=scores.get)

def next_move_policy(ps, counts, wins, losses):
    scores = {}
    for k, v in counts.items():
        weighted_pi = gamma * ps[k] / (1 + counts[k])
        vi = (wins.get(k, 0) - losses.get(k, 0)) / v if v > 0 else 0
        scores[k] = vi + weighted_pi
    return max(scores, key=scores.get)

print("✅ Đã cài đặt xong toàn bộ môi trường cốt lõi MCTS & Policy UCT!")

✅ Đã cài đặt xong toàn bộ môi trường cốt lõi MCTS & Policy UCT!


**Mixed MCTS v1**: Chỉ implement code áp dụng strong policy network cho bước selection của MCTS

In [15]:
# =============================================================================
# MIXED MCTS V1: STRONG POLICY CHO SELECTION + ROLLOUT NGẪU NHIÊN
# =============================================================================
def mix_mcts_v1(env, model, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    # 1. Dự đoán phân phối xác suất từ Strong Policy Network
    state = env.state.reshape(-1, 3, 3, 1)
    if env.turn == "X":
        action_probs = model(state, training=False).numpy()
    else:
        action_probs = model(-state, training=False).numpy()
    
    ps = {a: float(np.squeeze(action_probs)[a - 1]) for a in env.validinputs}
    
    # 2. Chạy các lượt mô phỏng Rollouts
    for _ in range(num_rollouts):
        # Bước 1: Selection (kết hợp Strong Policy)
        move = mix_select(env, ps, counts, wins, losses, temperature)
        # Bước 2: Expansion
        env_copy, done, reward = expand(env, move)
        # Bước 3: Simulation (Ngẫu nhiên thuần túy)
        reward = simulate(env_copy, done, reward)
        # Bước 4: Backpropagation
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_move_policy(ps, counts, wins, losses)

# Thử nghiệm nhanh 1 nước đi
test_env = ttt()
move_v1 = mix_mcts_v1(test_env, strong_model, num_rollouts=100)
print(f"🎯 [Mixed MCTS v1] Nước đi đề xuất cho bàn cờ trống: Ô số {move_v1}")

🎯 [Mixed MCTS v1] Nước đi đề xuất cho bàn cờ trống: Ô số 9


**Mixed MCTS v2**: Chỉ implement code áp dụng fast policy network cho bước rollout của MCTS

In [16]:
# =============================================================================
# MIXED MCTS V2: UCT CHO SELECTION + FAST POLICY CHO ROLLOUT
# =============================================================================
def simulate_fast_policy(env_copy, done, reward, model):
    if done:
        return reward
    while True:
        state = env_copy.state.reshape(-1, 3, 3, 1)
        if env_copy.turn == "X":
            probs = model(state, training=False).numpy().flatten()
        else:
            probs = model(-state, training=False).numpy().flatten()
            
        valid_moves = env_copy.validinputs
        valid_probs = np.array([probs[m - 1] for m in valid_moves])
        prob_sum = valid_probs.sum()
        
        if prob_sum > 0:
            valid_probs = valid_probs / prob_sum
            move = np.random.choice(valid_moves, p=valid_probs)
        else:
            move = random.choice(valid_moves)
            
        state, reward, done, info = env_copy.step(move)
        if done:
            return reward

def mix_mcts_v2(env, model, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    for _ in range(num_rollouts):
        # Bước 1: Selection (UCT chuẩn)
        move = select(env, counts, wins, losses, temperature)
        # Bước 2: Expansion
        env_copy, done, reward = expand(env, move)
        # Bước 3: Simulation (Dùng Fast Policy Network)
        reward = simulate_fast_policy(env_copy, done, reward, model)
        # Bước 4: Backpropagation
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_best_move(counts, wins, losses)

# Thử nghiệm nhanh 1 nước đi
test_env = ttt()
move_v2 = mix_mcts_v2(test_env, fast_model, num_rollouts=100)
print(f"🎯 [Mixed MCTS v2] Nước đi đề xuất cho bàn cờ trống: Ô số {move_v2}")

🎯 [Mixed MCTS v2] Nước đi đề xuất cho bàn cờ trống: Ô số 1


**Mixed MCTS v3**: Chỉ implement code, áp dụng cả 2 strong policy network cho bước selection của MCTS và fast policy network cho bước rollout của MCTS

In [17]:
# =============================================================================
# MIXED MCTS V3: STRONG POLICY CHO SELECTION + FAST POLICY CHO ROLLOUT
# =============================================================================
def mix_mcts_v3(env, strong_model, fast_model, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    # 1. Tính toán xác suất Tiên nghiệm từ Strong Policy Network
    state = env.state.reshape(-1, 3, 3, 1)
    if env.turn == "X":
        action_probs = strong_model(state, training=False).numpy()
    else:
        action_probs = strong_model(-state, training=False).numpy()
    
    ps = {a: float(np.squeeze(action_probs)[a - 1]) for a in env.validinputs}
    
    # 2. Rollout mô phỏng
    for _ in range(num_rollouts):
        # Bước 1: Selection (kết hợp Strong Policy)
        move = mix_select(env, ps, counts, wins, losses, temperature)
        # Bước 2: Expansion
        env_copy, done, reward = expand(env, move)
        # Bước 3: Simulation (Định hướng bởi Fast Policy)
        reward = simulate_fast_policy(env_copy, done, reward, fast_model)
        # Bước 4: Backpropagation
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_move_policy(ps, counts, wins, losses)

# Thử nghiệm nhanh 1 nước đi
test_env = ttt()
move_v3 = mix_mcts_v3(test_env, strong_model, fast_model, num_rollouts=100)
print(f"🎯 [Mixed MCTS v3 (AlphaGo)] Nước đi đề xuất cho bàn cờ trống: Ô số {move_v3}")

# Đánh giá thi đấu nhanh giữa Mixed MCTS v3 và MCTS thuần
print("\n⚔️ Chạy thử nghiệm đối đầu: Mixed MCTS v3 vs MCTS thuần (20 ván)...")
v3_wins, ties, mcts_wins = 0, 0, 0
for i in range(20):
    g_env = ttt()
    while True:
        # Lượt 1: v3 (X)
        act = mix_mcts_v3(g_env, strong_model, fast_model, num_rollouts=80)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == 1: v3_wins += 1
            else: ties += 1
            break
        # Lượt 2: MCTS truyền thống (O)
        act = mcts(g_env, num_rollouts=80)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == -1: mcts_wins += 1
            else: ties += 1
            break

print(f"🏆 Kết quả sau 20 ván: Mixed MCTS v3 Thắng {v3_wins} | Hòa {ties} | MCTS Thắng {mcts_wins}")

🎯 [Mixed MCTS v3 (AlphaGo)] Nước đi đề xuất cho bàn cờ trống: Ô số 2

⚔️ Chạy thử nghiệm đối đầu: Mixed MCTS v3 vs MCTS thuần (20 ván)...
🏆 Kết quả sau 20 ván: Mixed MCTS v3 Thắng 17 | Hòa 3 | MCTS Thắng 0


## **Phương pháp Reinforcement Learning áp dụng trong bài toán TicTacToe**

## Khởi tạo Q-learning values

In [ ]:
# =============================================================================
# 1. KHỞI TẠO BẢNG GIÁ TRỊ Q-TABLE CHO TICTACTOE
# =============================================================================
import numpy as np

# Bảng Q-table: 
# Key   : tuple 9 phần tử biểu diễn trạng thái bàn cờ
# Value : mảng numpy (9,) lưu Q-value của 9 nước đi tương ứng (ô 1 -> ô 9)
Q_table = {}

def get_q_values(state_key):
    """Lấy vector Q-value của trạng thái state_key; nếu chưa có thì khởi tạo vector 0."""
    if state_key not in Q_table:
        Q_table[state_key] = np.zeros(9, dtype=np.float32)
    return Q_table[state_key]

# Kiểm tra thử với bàn cờ trống
empty_board_key = tuple([0] * 9)
print("✅ Đã khởi tạo cấu trúc bảng Q_table!")
print(f"- Giá trị Q-values cho bàn cờ trống ban đầu:\n  {get_q_values(empty_board_key)}")
print(f"- Tổng số trạng thái đã lưu hiện tại: {len(Q_table)}")

✅ Đã khởi tạo cấu trúc bảng Q_table!
- Giá trị Q-values cho bàn cờ trống ban đầu:
  [0. 0. 0. 0. 0. 0. 0. 0. 0.]
- Tổng số trạng thái đã lưu hiện tại: 1


## Nhận diện môi trường 

In [ ]:
from copy import deepcopy
from math import sqrt, log
import pandas as pd

# --- THUẬT TOÁN PURE MCTS TỰ CHỨA ĐỂ THI ĐẤU ---
def mcts_expand(env, move):
    e = deepcopy(env)
    s, r, d, _ = e.step(move)
    return e, d, r

def mcts_simulate(env_copy, done, reward):
    if done: return reward
    while True:
        m = env_copy.sample()
        _, r, d, _ = env_copy.step(m)
        if d: return r

def mcts_backpropagate(env, move, reward, counts, wins, losses):
    counts[move] = counts.get(move, 0) + 1
    if reward == 1 and env.turn == "X": wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "O": wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "X": losses[move] = losses.get(move, 0) + 1
    elif reward == 1 and env.turn == "O": losses[move] = losses.get(move, 0) + 1
    return counts, wins, losses

def mcts_select(env, counts, wins, losses, temperature=1.414):
    for k in env.validinputs:
        if counts[k] == 0: return k
    N = sum(counts.values())
    scores = {k: (wins.get(k, 0) - losses.get(k, 0))/counts[k] + temperature * sqrt(log(N)/counts[k]) for k in env.validinputs}
    return max(scores, key=scores.get)

def pure_mcts_move(env, num_rollouts=60):
    if len(env.validinputs) == 1: return env.validinputs[0]
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    for _ in range(num_rollouts):
        m = mcts_select(env, counts, wins, losses)
        ec, d, r = mcts_expand(env, m)
        r = mcts_simulate(ec, d, r)
        counts, wins, losses = mcts_backpropagate(env, m, r, counts, wins, losses)
    scores = {k: (wins.get(k, 0) - losses.get(k, 0))/counts[k] if counts[k] > 0 else -float('inf') for k in counts.keys()}
    return max(scores, key=scores.get)

In [ ]:
# =============================================================================
# 3. NHẬN DIỆN MÔI TRƯỜNG & CHUẨN HÓA TRẠNG THÁI (CANONICAL STATE)
# =============================================================================
def get_canonical_state(env):
    """
    Chuẩn hóa góc nhìn trạng thái:
    - Lượt quân X: quân mình là 1, đối thủ là -1.
    - Lượt quân O: đảo dấu (-state), quân mình vẫn là 1, đối thủ là -1.
    """
    turn_mult = 1 if env.turn == "X" else -1
    return tuple(env.state * turn_mult)

# Thử nghiệm nhận diện môi trường
test_env = ttt()
test_env.reset()
test_env.step(5)  # X đánh vào ô tâm (ô số 5)

print("🎮 BÀN CỜ MẪU (X vừa đánh ô 5, đến lượt O):")
print(test_env.state.reshape(3, 3)[::-1])
print(f"- Lượt đi hiện tại           : Quân '{test_env.turn}'")
print(f"- Các ô trống hợp lệ         : {test_env.validinputs}")
print(f"- Vector trạng thái gốc      : {tuple(test_env.state)}")
print(f"- Trạng thái chuẩn hóa cho O : {get_canonical_state(test_env)}")

🎮 BÀN CỜ MẪU (X vừa đánh ô 5, đến lượt O):
[[0 0 0]
 [0 1 0]
 [0 0 0]]
- Lượt đi hiện tại           : Quân 'O'
- Các ô trống hợp lệ         : [1, 2, 3, 4, 6, 7, 8, 9]
- Vector trạng thái gốc      : (0, 0, 0, 0, 1, 0, 0, 0, 0)
- Trạng thái chuẩn hóa cho O : (0, 0, 0, 0, -1, 0, 0, 0, 0)


## Khởi tạo tác nhân

Áp dụng công thức tối ưu phần thưởng của Q learning table, ta khởi tạo tác nhân tự học chơi TicTacToe và quan sát Q-value của mô hình

In [ ]:
# =============================================================================
# 2. KHỞI TẠO TÁC NHÂN Q-LEARNING VỚI CHIẾN LƯỢC EPSILON-GREEDY
# =============================================================================
import random

def select_action(env, state_key, epsilon=0.1):
    """
    Lựa chọn nước đi theo chiến lược Epsilon-Greedy:
    - Với xác suất epsilon: Khám phá (Exploration) -> đi ngẫu nhiên ô hợp lệ.
    - Với xác suất 1 - epsilon: Khai thác (Exploitation) -> chọn ô có Q-value cao nhất trong các ô hợp lệ.
    """
    valid_moves = env.validinputs
    if len(valid_moves) == 0:
        return None
    if len(valid_moves) == 1:
        return valid_moves[0]
        
    # Khám phá ngẫu nhiên
    if np.random.rand() < epsilon:
        return random.choice(valid_moves)
        
    # Khai thác Q-table
    q_vals = get_q_values(state_key)
    valid_q = {m: q_vals[m - 1] for m in valid_moves}
    max_q = max(valid_q.values())
    
    # Nếu có nhiều nước đi cùng đạt max Q, chọn ngẫu nhiên giữa các nước đó
    best_moves = [m for m, val in valid_q.items() if val == max_q]
    return random.choice(best_moves)

def q_agent_move(env):
    """Quyết định nước đi thuần túy cho thi đấu (Greedy tuyệt đối, epsilon = 0.0)."""
    state_key = get_canonical_state(env)
    return select_action(env, state_key, epsilon=0.0)

print("✅ Đã khởi tạo tác nhân Q-learning với chiến lược Epsilon-Greedy thành công!")

✅ Đã khởi tạo tác nhân Q-learning với chiến lược Epsilon-Greedy thành công!


## Định nghĩa phần thưởng

In [ ]:
# =============================================================================
# 4. ĐỊNH NGHĨA PHẦN THƯỞNG & HÀM CẬP NHẬT PHƯƠNG TRÌNH BELLMAN
# =============================================================================
"""
Quy ước phần thưởng (Zero-Sum Reward):
- Thắng ván cờ  : +1.0
- Thua ván cờ   : -1.0
- Hòa ván cờ    :  0.0
- Nước đi thường:  0.0

Công thức cập nhật Bellman Equation (Temporal Difference Q-learning):
Q(s, a) = Q(s, a) + lr * [ Target - Q(s, a) ]
Trong đó:
- Target nước kết thúc : Reward (+1, -1, hoặc 0)
- Target nước trung gian: Reward + gamma * max_a' Q(s', a')
"""

def update_q_value(state_key, action, target, lr):
    """Cập nhật giá trị Q cho cặp trạng thái - hành động (s, a)."""
    q_vals = get_q_values(state_key)
    current_q = q_vals[action - 1]
    q_vals[action - 1] += lr * (target - current_q)

print("✅ Đã thiết lập cấu trúc phần thưởng và hàm cập nhật Bellman Equation!")

✅ Đã thiết lập cấu trúc phần thưởng và hàm cập nhật Bellman Equation!


## Training theo Episode

```python
# the learning rate
lr=0.01
# discount rate
gamma=0.95
# parameters to control exploration
max_exp=0.9
min_exp=0.1
# maximum steps in a game
max_steps=50
# number of episodes to train Q-values
max_episode=10000
```

In [ ]:
# =============================================================================
# 5. VÒNG LẶP HUẤN LUYỆN Q-LEARNING QUA CÁC EPISODE (SELF-PLAY)
# =============================================================================
import time

# Thiết lập các siêu tham số huấn luyện (theo khung tham số chương 12)
lr = 0.1               # Tốc độ học (learning rate)
gamma = 0.8            # Hệ số chiết khấu phần thưởng tương lai
max_exp = 0.9           # Mức khám phá ban đầu (90% ngẫu nhiên)
min_exp = 0.1          # Mức khám phá cuối cùng (5% ngẫu nhiên)
max_episode = 100000     # Số ván cờ huấn luyện
save_interval = 2000    # Chu kỳ báo cáo kết quả

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN Q-LEARNING QUA {max_episode} VÁN TỰ ĐẤU (SELF-PLAY)...")
t_start = time.time()

stats_x_wins = 0
stats_o_wins = 0
stats_ties = 0

# Khởi tạo môi trường bàn cờ TicTacToe
env = ttt()

for ep in range(1, max_episode + 1):
    env.reset()
    # Epsilon suy giảm tuyến tính theo tiến trình học
    epsilon = max_exp - (max_exp - min_exp) * (ep / max_episode)
    trajectory = []
    
    while True:
        state_key = get_canonical_state(env)
        player_turn = env.turn
        
        # Chọn nước đi theo Epsilon-Greedy
        action = select_action(env, state_key, epsilon)
        trajectory.append((state_key, action, player_turn))
        
        state, reward, done, _ = env.step(action)
        if done:
            break
            
    # Ghi nhận kết quả ván cờ
    if reward == 1:
        stats_x_wins += 1
    elif reward == -1:
        stats_o_wins += 1
    else:
        stats_ties += 1

    # Cập nhật Bellman lan truyền ngược theo quỹ đạo ván đấu (TD-Learning)
    final_reward_x = reward  # 1 nếu X thắng, -1 nếu O thắng, 0 nếu hòa
    
    for i in reversed(range(len(trajectory))):
        s_k, act, p_turn = trajectory[i]
        # Phần thưởng đối với người chơi ở lượt đó
        p_reward = final_reward_x if p_turn == "X" else -final_reward_x
        
        # Nước đi dẫn đến kết thúc ván
        if i >= len(trajectory) - 2:
            target = p_reward
        else:
            # Trạng thái kế tiếp của chính người chơi đó (sau khi đối thủ đã đáp trả)
            next_s_k, _, _ = trajectory[i + 2]
            next_max_q = np.max(get_q_values(next_s_k))
            target = p_reward + gamma * next_max_q
            
        update_q_value(s_k, act, target, lr)
        
    # Báo cáo tiến độ định kỳ
    if ep % save_interval == 0 or ep == max_episode:
        elapsed = time.time() - t_start
        print(f"Episode {ep:5d}/{max_episode} ({elapsed:5.1f}s) | Epsilon: {epsilon:.3f} | "
              f"Số trạng thái trong Q: {len(Q_table):4d} | X Thắng: {stats_x_wins:4d} | O Thắng: {stats_o_wins:4d} | Hòa: {stats_ties:4d}")
        stats_x_wins, stats_o_wins, stats_ties = 0, 0, 0

total_time = time.time() - t_start
print(f"\n🎉 HOÀN THÀNH HUẤN LUYỆN {max_episode} VÁN TRONG {total_time:.2f} GIÂY!")
print(f"📊 Tổng số trạng thái bàn cờ đã được khám phá và tối ưu: {len(Q_table)} trạng thái.")

🚀 BẮT ĐẦU HUẤN LUYỆN Q-LEARNING QUA 100000 VÁN TỰ ĐẤU (SELF-PLAY)...
Episode  2000/100000 (  0.4s) | Epsilon: 0.884 | Số trạng thái trong Q: 4520 | X Thắng: 1150 | O Thắng:  613 | Hòa:  237
Episode  4000/100000 (  0.7s) | Epsilon: 0.868 | Số trạng thái trong Q: 4520 | X Thắng: 1163 | O Thắng:  594 | Hòa:  243
Episode  6000/100000 (  1.0s) | Epsilon: 0.852 | Số trạng thái trong Q: 4520 | X Thắng: 1112 | O Thắng:  612 | Hòa:  276
Episode  8000/100000 (  1.3s) | Epsilon: 0.836 | Số trạng thái trong Q: 4520 | X Thắng: 1147 | O Thắng:  587 | Hòa:  266
Episode 10000/100000 (  1.5s) | Epsilon: 0.820 | Số trạng thái trong Q: 4520 | X Thắng: 1111 | O Thắng:  623 | Hòa:  266
Episode 12000/100000 (  1.8s) | Epsilon: 0.804 | Số trạng thái trong Q: 4520 | X Thắng: 1149 | O Thắng:  603 | Hòa:  248
Episode 14000/100000 (  2.0s) | Epsilon: 0.788 | Số trạng thái trong Q: 4520 | X Thắng: 1131 | O Thắng:  611 | Hòa:  258
Episode 16000/100000 (  2.3s) | Epsilon: 0.772 | Số trạng thái trong Q: 4520 | X Thắ

Lưu lại các model 1 theo số lượng episode huấn luyện khác nhau

In [ ]:
# =============================================================================
# 6. LƯU VÀ TẢI BẢNG Q-TABLE TỪ FILE
# =============================================================================
import os
import pickle

SAVE_DIR = "files"
os.makedirs(SAVE_DIR, exist_ok=True)
Q_TABLE_FILE = os.path.join(SAVE_DIR, "q_table_ttt_100000.p")

# Lưu Q-table ra file
with open(Q_TABLE_FILE, "wb") as fp:
    pickle.dump(Q_table, fp)
print(f"💾 Đã lưu Q-table thành công tại: {Q_TABLE_FILE} ({os.path.getsize(Q_TABLE_FILE)/1024:.1f} KB)")

def load_q_table(filepath):
    """Hàm tải lại bảng Q-table từ đĩa."""
    global Q_table
    with open(filepath, "rb") as fp:
        Q_table = pickle.load(fp)
    print(f"📂 Đã nạp thành công Q-table ({len(Q_table)} trạng thái) từ {filepath}!")

💾 Đã lưu Q-table thành công tại: files\q_table_ttt_100000.p (900.6 KB)


# Kiểm thử mô hình Q-learning

Cho Q-learning đánh với Pure MCTS

In [ ]:
# =============================================================================
# 7. ĐẤU TRƯỜNG ĐỐI ĐẦU: Q-LEARNING AGENT VS PURE MCTS (100 VÁN)
# =============================================================================

# --- THI ĐẤU 100 VÁN ---
total_matches = 1000
half = total_matches // 2
q_wins, ties, mcts_wins = 0, 0, 0

print(f"⚔️ Bắt đầu loạt trận đối đầu ({total_matches} ván)...")

# 50 ván Q-Agent đi trước (X)
for _ in range(half):
    g_env = ttt()
    while True:
        # Lượt 1: Q-Agent (X)
        act = q_agent_move(g_env)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == 1: q_wins += 1
            else: ties += 1
            break
            
        # Lượt 2: Pure MCTS (O)
        act = pure_mcts_move(g_env, num_rollouts=60)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == -1: mcts_wins += 1
            else: ties += 1
            break

# 50 ván Q-Agent đi sau (O)
for _ in range(half):
    g_env = ttt()
    while True:
        # Lượt 1: Pure MCTS (X)
        act = pure_mcts_move(g_env, num_rollouts=60)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == 1: mcts_wins += 1
            else: ties += 1
            break
            
        # Lượt 2: Q-Agent (O)
        act = q_agent_move(g_env)
        _, rew, done, _ = g_env.step(act)
        if done:
            if rew == -1: q_wins += 1
            else: ties += 1
            break

# --- TỔNG KẾT BẢNG SỐ LIỆU ---
df_results = pd.DataFrame([{
    "Tổng số ván": total_matches,
    "Q-Agent Thắng": q_wins,
    "Hòa": ties,
    "Pure MCTS Thắng": mcts_wins,
    "Tỷ lệ Q Thắng (%)": f"{q_wins/total_matches*100:.1f}%",
    "Tỷ lệ Hòa (%)": f"{ties/total_matches*100:.1f}%",
    "Tỷ lệ Bất bại của Q (%)": f"{(q_wins + ties)/total_matches*100:.1f}%"
}])

print("\n📊 BẢNG TỔNG KẾT ĐỐI ĐẦU: Q-LEARNING VS PURE MCTS:")
display(df_results)

⚔️ Bắt đầu loạt trận đối đầu (1000 ván)...

📊 BẢNG TỔNG KẾT ĐỐI ĐẦU: Q-LEARNING VS PURE MCTS:


,Tổng số ván,Q-Agent Thắng,Hòa,Pure MCTS Thắng,Tỷ lệ Q Thắng (%),Tỷ lệ Hòa (%),Tỷ lệ Bất bại của Q (%)
0,1000,503,497,0,50.3%,49.7%,100.0%


## **Phương pháp Rule-based áp dụng trong bài toán TicTacToe**
Áp dụng phương pháp nhìn trước nước đi (Look-Ahead Search) 1 bước, 2 bước và 3 bước cho trò chơi Tic Tac Toe theo kiến trúc của AlphaGo Simplified.

## Nạp môi trường và framework

In [ ]:
import turtle as t
import numpy as np
import time
import random

# Define an action_space helper class
class action_space:
    def __init__(self, n):
        self.n = n
    
# Define an obervation_space helper class    
class observation_space:
    def __init__(self, n):
        self.shape = (n,)

class ttt():
    def __init__(self): 
        # use the helper action_space class
        self.action_space=action_space(9)
        # use the helper observation_space class
        self.observation_space=observation_space(9)
        self.info=""         
        # Create a dictionary to map cell number to coordinates
        self.cellcenter = {1:(-200,-200), 2:(0,-200), 3:(200,-200),
                    4:(-200,0), 5:(0,0), 6:(200,0),
                    7:(-200,200), 8:(0,200), 9:(200,200)} 
        # set the game to the initial state
        self.reset()
        
    # sample() function: returns a random move
    def sample(self):
        return random.choice(self.validinputs)   

    def reset(self):  
        # The X player moves first
        self.turn = "X"
        # Count how many rounds played
        self.rounds = 1
        # Create a list of valid moves
        self.validinputs = list(self.cellcenter.keys())
        # Create a dictionary of moves made by each player
        self.occupied = {"X":[],"O":[]}
        # Tracking the state
        self.state=np.array([0,0,0,0,0,0,0,0,0])
        self.done=False
        self.reward=0     
        return self.state        
        
    # step() function: place piece on board and update state
    def step(self, inp):
        # Add the move to the occupied list 
        self.occupied[self.turn].append(inp)
        # update the state: X is 1 and O is -1
        self.state[int(inp)-1]=2*(self.turn=="X")-1
        # Disallow the move in future rounds
        self.validinputs.remove(inp) 
        # check if the player has won the game
        if self.win_game() == True:
            self.done=True
            # reward is 1 if X won; -1 if O won
            self.reward=2*(self.turn=="X")-1
            self.validinputs=[]
        # If all cellls are occupied and no winner, it's a tie
        elif self.rounds == 9:
            self.done=True
            self.reward=0
            self.validinputs=[]
        else:
            # Counting rounds
            self.rounds += 1
            # Give the turn to the other player
            if self.turn == "X":
                self.turn = "O"
            else:
                self.turn = "X"             
        return self.state, self.reward, self.done, self.info
                    
    # Determine if a player has won the game
    def win_game(self):
        lst = self.occupied[self.turn]
        if 1 in lst and 2 in lst and 3 in lst:
            return True
        elif 4 in lst and 5 in lst and 6 in lst:
            return True        
        elif 7 in lst and 8 in lst and 9 in lst:
            return True        
        elif 1 in lst and 4 in lst and 7 in lst:
            return True
        elif 2 in lst and 5 in lst and 8 in lst:
            return True
        elif 3 in lst and 6 in lst and 9 in lst:
            return True
        elif 1 in lst and 5 in lst and 9 in lst:
            return True
        elif 3 in lst and 5 in lst and 7 in lst:
            return True
        else:
            return False

In [ ]:
import numpy as np
from copy import deepcopy
import random
import time

# Nạp môi trường cờ ca-rô TicTacToe (bản ttt_simple_env thuần logic, tốc độ cao không GUI)
# from utils.ttt_simple_env import ttt

# Định nghĩa người chơi đánh ngẫu nhiên
def ttt_random(env):
    return env.sample()

# Hàm mô phỏng 1 ván đấu hoàn chỉnh giữa 2 đấu thủ (player1 đi trước, player2 đi sau)
def one_ttt_game(player1, player2):
    env = ttt()
    env.reset()     
    while True:    
        # Lượt player1 (quân X)
        action = player1(env)  
        state, reward, done, _ = env.step(action)
        if done:
            break
            
        # Lượt player2 (quân O)
        action = player2(env)  
        state, reward, done, _ = env.step(action)
        if done:
            break            
    return reward  # 1: player1 thắng, -1: player2 thắng, 0: hòa


## Xem trước 1 bước trong trò chơi

In [ ]:
def ttt_think1(env):
    """
    AI xem trước 1 bước:
    - Duyệt qua từng nước đi hợp lệ hiện tại.
    - Dùng deepcopy để giả định nước đi.
    - Nếu nước đi giúp thắng ngay (done và reward = 1 hoặc -1), chọn ngay nước đó.
    - Nếu không có nước nào thắng ngay, chọn ngẫu nhiên.
    """
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, _ = env_copy.step(m)
        if done and abs(reward) == 1:
            return m
    return env.sample()

# Thử nghiệm đánh giá hiệu quả: ttt_think1 đấu với ttt_random (1000 ván)
print("Đang chạy thử nghiệm ttt_think1 vs ttt_random (1000 ván)...\n")
results_think1 = []
for i in range(1000):
    if i % 2 == 0:
        # AI đi trước (X)
        r = one_ttt_game(ttt_think1, ttt_random)
        results_think1.append(r)
    else:
        # AI đi sau (O) - đảo dấu để 1 luôn đại diện cho AI thắng
        r = one_ttt_game(ttt_random, ttt_think1)
        results_think1.append(-r)

wins = results_think1.count(1)
losses = results_think1.count(-1)
ties = results_think1.count(0)
print(f"Kết quả sau 1000 ván (think1 vs random):")
print(f" - AI thắng : {wins} ({wins/10:.1f}%)")
print(f" - AI thua  : {losses} ({losses/10:.1f}%)")
print(f" - Hòa      : {ties} ({ties/10:.1f}%)")


## Xem trước 2 bước trong trò chơi

In [ ]:
def ttt_think2(env):
    """
    AI xem trước 2 bước:
    1. Kiểm tra nếu có nước thắng ngay (1 bước) -> đánh ngay.
    2. Nếu không, dự đoán 2 bước tiếp theo (m1 của ta, m2 của đối thủ):
       Nếu đối thủ thắng ở lượt m2 -> chặn ngay hiểm họa bằng cách đánh vào m2.
    3. Nếu không có hiểm họa -> đi ngẫu nhiên.
    """
    # 1. Tìm nước thắng ngay cho bản thân
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, _ = env_copy.step(m)
        if done and abs(reward) == 1:
            return m

    # 2. Dự đoán 2 bước để chặn đối thủ
    for m1 in env.validinputs:
        for m2 in env.validinputs:
            if m1 != m2:
                env_copy = deepcopy(env)
                s, r, done, _ = env_copy.step(m1)
                s, r, done, _ = env_copy.step(m2)
                # Nếu đối thủ thắng ở bước m2 -> phải chặn ngay ô m2
                if done and r != 0:
                    return m2

    # 3. Nếu không có đe dọa, chọn ngẫu nhiên
    return env.sample()

# Thử nghiệm đánh giá hiệu quả: ttt_think2 đấu với ttt_think1 (1000 ván)
print("Đang chạy thử nghiệm ttt_think2 vs ttt_think1 (1000 ván)...\n")
results_think2 = []
for i in range(1000):
    if i % 2 == 0:
        r = one_ttt_game(ttt_think2, ttt_think1)
        results_think2.append(r)
    else:
        r = one_ttt_game(ttt_think1, ttt_think2)
        results_think2.append(-r)

wins = results_think2.count(1)
losses = results_think2.count(-1)
ties = results_think2.count(0)
print(f"Kết quả sau 1000 ván (think2 vs think1):")
print(f" - think2 thắng : {wins} ({wins/10:.1f}%)")
print(f" - think2 thua  : {losses} ({losses/10:.1f}%)")
print(f" - Hòa          : {ties} ({ties/10:.1f}%)")


## Xem trước 3 bước trong trò chơi

In [ ]:
def ttt_think3(env):
    """
    AI xem trước 3 bước:
    1. Tìm nước thắng ngay (1 bước).
    2. Chặn nước thắng của đối thủ (2 bước).
    3. Xem trước 3 bước: Ta (m1) -> Đối thủ (m2) -> Ta (m3).
       Nếu m3 thắng, lưu lại nước m1. Chọn nước m1 tạo ra nhiều kịch bản thắng nhất
       (thế công đôi / double attack - đối thủ chỉ có thể đỡ được 1 đường).
    4. Nếu không có, đi ngẫu nhiên.
    """
    # 1. Tìm nước thắng ngay
    for m in env.validinputs:
        env_copy = deepcopy(env)
        state, reward, done, _ = env_copy.step(m)
        if done and abs(reward) == 1:
            return m

    # 2. Chặn đối thủ thắng ở bước 2
    for m1 in env.validinputs:
        for m2 in env.validinputs:
            if m1 != m2:
                env_copy = deepcopy(env)
                s, r, done, _ = env_copy.step(m1)
                s, r, done, _ = env_copy.step(m2)
                if done and r != 0:
                    return m2

    # 3. Xem trước 3 bước để tìm nước tạo ra thế công đôi (Double Attack)
    w3 = []
    for m1 in env.validinputs:
        for m2 in env.validinputs:
            for m3 in env.validinputs:
                if m1 != m2 and m1 != m3 and m2 != m3:
                    env_copy = deepcopy(env)
                    s, r, done, _ = env_copy.step(m1)
                    s, r, done, _ = env_copy.step(m2)
                    s, r, done, _ = env_copy.step(m3)
                    if done and r != 0:
                        w3.append(m1)

    # Chọn nước m1 có số lần dẫn đến chiến thắng nhiều nhất
    if len(w3) > 0:
        return max(w3, key=w3.count)

    # 4. Nếu không có kịch bản nào, đi ngẫu nhiên
    return env.sample()

# Thử nghiệm đánh giá hiệu quả: ttt_think3 đấu với ttt_think2 (1000 ván)
print("Đang chạy thử nghiệm ttt_think3 vs ttt_think2 (1000 ván)...\n")
results_think3 = []
for i in range(1000):
    if i % 2 == 0:
        r = one_ttt_game(ttt_think3, ttt_think2)
        results_think3.append(r)
    else:
        r = one_ttt_game(ttt_think2, ttt_think3)
        results_think3.append(-r)

wins = results_think3.count(1)
losses = results_think3.count(-1)
ties = results_think3.count(0)
print(f"Kết quả sau 1000 ván (think3 vs think2):")
print(f" - think3 thắng : {wins} ({wins/10:.1f}%)")
print(f" - think3 thua  : {losses} ({losses/10:.1f}%)")
print(f" - Hòa          : {ties} ({ties/10:.1f}%)")


## Rule-based AI dựa vào "Human Problem Solving" của Newell & Simon (1972)

**Bản chất:** Rule-based AI không tự "học" từ dữ liệu mà thực thi hệ luật chuyên gia do con người thiết lập sẵn. Năm 1972, hai nhà khoa học tiên phong về AI là **Allen Newell** và **Herbert Simon** đã chứng minh hệ thống **8 quy tắc ưu tiên tuần tự** giúp người chơi **BẤT BẠI 100% (Thua = 0%)** trong Tic Tac Toe:

1. **Win (Thắng ngay):** Nếu có 2 quân thẳng hàng, đi ô thứ 3 để thắng ngay.
2. **Block (Chặn ngay):** Nếu đối thủ có 2 quân thẳng hàng, đi ô thứ 3 để chặn đối thủ.
3. **Fork (Tạo thế song sát):** Đánh vào ô tạo ra 2 đường thắng tiềm năng cùng lúc (buộc đối thủ chỉ đỡ được 1 đường).
4. **Block Fork (Phá thế song sát đối thủ):** Tạo phản công buộc đối thủ phải đỡ (nước đỡ không tạo Fork), hoặc chặn trực tiếp ô Fork của đối thủ.
5. **Center (Chiếm tâm):** Đánh vào ô chính giữa (ô số 5).
6. **Opposite Corner (Góc đối diện):** Nếu đối thủ chiếm một góc, đánh vào góc đối diện trên đường chéo.
7. **Empty Corner (Chiếm góc trống):** Đánh vào bất kỳ góc nào còn trống (1, 3, 7, 9).
8. **Empty Side (Chiếm cạnh trống):** Đánh vào ô cạnh giữa còn trống (2, 4, 6, 8).

In [ ]:
# 8 đường thắng chuẩn trong Tic Tac Toe (theo tọa độ bàn cờ 1-9)
LINES = [
    (1, 2, 3), (4, 5, 6), (7, 8, 9),  # 3 hàng ngang
    (1, 4, 7), (2, 5, 8), (3, 6, 9),  # 3 hàng dọc
    (1, 5, 9), (3, 5, 7)              # 2 đường chéo
]

def count_threats(state, piece):
    """Đếm số đường có 2 quân của 'piece' và 1 ô trống (mối đe dọa thắng)"""
    threats = 0
    for a, b, c in LINES:
        line = [state[a - 1], state[b - 1], state[c - 1]]
        if line.count(piece) == 2 and line.count(0) == 1:
            threats += 1
    return threats

def ttt_newell_simon(env):
    """
    Rule-based AI áp dụng triệt để 8 luật ưu tiên của Newell & Simon (1972)
    Đảm bảo 100% không thua (chỉ Thắng hoặc Hòa).
    """
    my_piece = 1 if env.turn == 'X' else -1
    opp_piece = -my_piece
    valid = env.validinputs

    # 1. RULE 1: WIN (Nếu có cơ hội thắng ngay trong lượt này)
    for m in valid:
        tmp = env.state.copy()
        tmp[m - 1] = my_piece
        for a, b, c in LINES:
            if tmp[a - 1] == tmp[b - 1] == tmp[c - 1] == my_piece:
                return m

    # 2. RULE 2: BLOCK (Nếu đối thủ có thể thắng ở lượt tới -> chặn ngay)
    for m in valid:
        tmp = env.state.copy()
        tmp[m - 1] = opp_piece
        for a, b, c in LINES:
            if tmp[a - 1] == tmp[b - 1] == tmp[c - 1] == opp_piece:
                return m

    # 3. RULE 3: FORK (Tạo thế song sát - tạo ra >= 2 đường thắng tiềm năng)
    for m in valid:
        tmp = env.state.copy()
        tmp[m - 1] = my_piece
        if count_threats(tmp, my_piece) >= 2:
            return m

    # 4. RULE 4: BLOCK FORK (Hóa giải / Phá thế song sát của đối thủ)
    opp_fork_cells = []
    for m in valid:
        tmp = env.state.copy()
        tmp[m - 1] = opp_piece
        if count_threats(tmp, opp_piece) >= 2:
            opp_fork_cells.append(m)

    if opp_fork_cells:
        # Cách 4A: Phản công ép đối thủ phải đỡ, miễn là ô đỡ không nằm trong opp_fork_cells
        forced_move = None
        for m in valid:
            tmp = env.state.copy()
            tmp[m - 1] = my_piece
            for a, b, c in LINES:
                line_cells = [a, b, c]
                line_vals = [tmp[x - 1] for x in line_cells]
                if line_vals.count(my_piece) == 2 and line_vals.count(0) == 1:
                    defense_cell = line_cells[line_vals.index(0)]
                    if defense_cell not in opp_fork_cells:
                        forced_move = m
                        break
            if forced_move:
                break
        if forced_move:
            return forced_move

        # Cách 4B: Nếu không thể phản công an toàn -> chiếm trực tiếp ô Fork của đối thủ
        return opp_fork_cells[0]

    # 5. RULE 5: CENTER (Chiếm ô trung tâm 5)
    if 5 in valid:
        return 5

    # 6. RULE 6: OPPOSITE CORNER (Chiếm góc đối diện góc đối thủ đã chiếm)
    opp_corner_pairs = [(1, 9), (9, 1), (3, 7), (7, 3)]
    for opp_c, my_c in opp_corner_pairs:
        if env.state[opp_c - 1] == opp_piece and my_c in valid:
            return my_c

    # 7. RULE 7: EMPTY CORNER (Chiếm góc trống)
    corners = [c for c in [1, 3, 7, 9] if c in valid]
    if corners:
        return corners[0]

    # 8. RULE 8: EMPTY SIDE (Chiếm cạnh trống)
    sides = [s for s in [2, 4, 6, 8] if s in valid]
    if sides:
        return sides[0]

    return valid[0]

# =============================================================================
# THỬ NGHIỆM ĐÁNH GIÁ: ttt_newell_simon ĐẤU VỚI ttt_random (1000 VÁN)
# =============================================================================
print("Đang chạy thử nghiệm Newell & Simon AI vs Random AI (1000 ván)...\n")
results_ns = []
for i in range(1000):
    if i % 2 == 0:
        # AI đi trước (X)
        r = one_ttt_game(ttt_newell_simon, ttt_random)
        results_ns.append(r)
    else:
        # AI đi sau (O) - đảo dấu để 1 luôn đại diện cho AI thắng
        r = one_ttt_game(ttt_random, ttt_newell_simon)
        results_ns.append(-r)

wins = results_ns.count(1)
losses = results_ns.count(-1)
ties = results_ns.count(0)
print(f"Kết quả sau 1000 ván (Newell & Simon vs Random):")
print(f" - Newell & Simon thắng : {wins} ({wins/10:.1f}%)")
print(f" - Newell & Simon thua  : {losses} ({losses/10:.1f}%)")
print(f" - Hòa                  : {ties} ({ties/10:.1f}%)")
if losses == 0:
    print("\n===> KẾT LUẬN: Đúng như lý thuyết, Newell & Simon AI BẤT BẠI (Tỷ lệ thua = 0%)!")

## **Phương pháp MiniMax áp dụng trong bài toán TicTacToe**

## 1. Môi trường MiniMax & Thuật toán Nền tảng (Chapter 5)

Trong Zero-Sum Game (trò chơi tổng bằng 0), ta áp dụng công thức NegaMax: `my_payoff = -opponent_payoff`. Điểm số của đối thủ chính là giá trị đối nghịch với điểm của ta.
- Trạng thái kết thúc: Nếu người chơi vừa đi thắng $\to$ Ta bị $-1$ điểm; Nếu hòa $\to$ $0$ điểm.
- Người chơi chọn nước đi tối đa hóa điểm số (`best_payoff = max(my_payoff)`).

In [ ]:
import random
import time
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Nạp môi trường từ thư mục utils nội bộ
from utils.coin_simple_env import coin_game
from utils.ttt_simple_env import ttt

# =============================================================================
# 1. HÀM ĐẤU THỦ NGẪU NHIÊN & MÔ PHỎNG VÁN CỜ
# =============================================================================
def ttt_random(env):
    """Đấu thủ chọn nước đi ngẫu nhiên từ các ô hợp lệ."""
    return env.sample()

def one_ttt_game(player1, player2):
    """Mô phỏng 1 ván cờ chuẩn giữa 2 đấu thủ (player1 cầm X đi trước, player2 cầm O đi sau)."""
    env = ttt()
    env.reset()
    while True:
        # Lượt player1 (quân X)
        action = player1(env)
        _, reward, done, _ = env.step(action)
        if done:
            return reward  # 1: X thắng, -1: O thắng, 0: hòa
            
        # Lượt player2 (quân O)
        action = player2(env)
        _, reward, done, _ = env.step(action)
        if done:
            return reward

# =============================================================================
# 2. THUẬT TOÁN MINIMAX ĐỆ QUY CHO TICTACTOE (Theo Chapter 5 - AlphaGo Simplified)
# =============================================================================
memo_payoff = {}

def maximized_payoff(env, reward, done):
    """
    Hàm đệ quy tính toán kết quả tốt nhất mà người chơi hiện tại có thể đạt được
    sau khi người chơi trước vừa thực hiện nước đi (Chapter 5).
    Tích hợp Transposition Table (memoization) tối ưu tốc độ.
    """
    if done:
        return -1 if reward != 0 else 0

    key = (tuple(env.state), env.turn)
    if key in memo_payoff:
        return memo_payoff[key]

    best_payoff = -2
    for m in env.validinputs:
        env_copy = deepcopy(env)
        _, rew, d, _ = env_copy.step(m)
        opponent_payoff = maximized_payoff(env_copy, rew, d)
        my_payoff = -opponent_payoff
        if my_payoff > best_payoff:
            best_payoff = my_payoff
            if best_payoff == 1:
                break

    memo_payoff[key] = best_payoff
    return best_payoff

def MiniMax_TTT(env):
    """Hàm chọn nước đi tối ưu cho Tic Tac Toe bằng MiniMax thuần túy (Chapter 5)."""
    wins, ties, losses = [], [], []
    for m in env.validinputs:
        env_copy = deepcopy(env)
        _, rew, done, _ = env_copy.step(m)
        if done and rew != 0:
            return m
        my_payoff = -maximized_payoff(env_copy, rew, done)
        if my_payoff == 1:
            wins.append(m)
        elif my_payoff == 0:
            ties.append(m)
        else:
            losses.append(m)

    if len(wins) > 0:
        return random.choice(wins)
    elif len(ties) > 0:
        return random.choice(ties)
    return env.sample()

print("✅ Đã nạp môi trường coin_game và ttt thành công!")
print("✅ Đã định nghĩa: ttt_random, one_ttt_game, maximized_payoff, MiniMax_TTT.")


✅ Đã nạp môi trường coin_game và ttt thành công!
✅ Đã định nghĩa: ttt_random, one_ttt_game, maximized_payoff, MiniMax_TTT.


## 2. Tỉa Cây Theo Các Phương Pháp (Depth Pruning & Alpha-Beta Pruning)

Khi bàn cờ còn trống, cây tìm kiếm Tic Tac Toe có tới $9! = 362.880$ nhánh lá. Để giảm tải tính toán mà vẫn đảm bảo nước cờ thông minh, ta có 2 kỹ thuật tỉa cây kinh điển trong *AlphaGo Simplified*:
1. **Cắt tỉa độ sâu (Depth Pruning - Chapter 5)**: Giới hạn độ sâu tối đa $d$. Nếu chưa hết ván khi đạt ngưỡng độ sâu thì tạm coi là Hòa (0 điểm).
2. **Cắt tỉa Alpha-Beta (Alpha-Beta Pruning - Chapter 6)**: Dùng 2 ngưỡng $\alpha$ (giá trị tối thiểu người chơi hiện tại chắc chắn đạt được) và $\beta$ (giá trị tối đa đối thủ cho phép đạt được). Nhánh nào có $\alpha \ge \beta$ sẽ bị cắt bỏ ngay lập tức mà **không làm mất tính chính xác toán học**.

In [ ]:
# =============================================================================
# 1. CẮT TỈA ĐỘ SÂU (DEPTH PRUNING - CHAPTER 5)
# =============================================================================
def max_payoff(env, reward, done, depth):
    """Hàm đệ quy giới hạn độ sâu (Depth Pruning) theo Chapter 5."""
    if done:
        return -1 if reward != 0 else 0
    if depth == 0:
        return 0  # Heuristic tạm thời: Coi như hòa nếu chạm ngưỡng độ sâu
        
    best_payoff = -2
    for m in env.validinputs:
        env_copy = deepcopy(env)
        _, rew, d, _ = env_copy.step(m)
        opponent_payoff = max_payoff(env_copy, rew, d, depth - 1)
        my_payoff = -opponent_payoff
        if my_payoff > best_payoff:
            best_payoff = my_payoff
            if best_payoff == 1:
                break
    return best_payoff

def MiniMax_depth(env, depth=2):
    """Hàm chọn nước đi MiniMax cắt tỉa độ sâu (mặc định d=2)."""
    wins, ties, losses = [], [], []
    for m in env.validinputs:
        env_copy = deepcopy(env)
        _, rew, done, _ = env_copy.step(m)
        if done and rew != 0:
            return m
        my_payoff = -max_payoff(env_copy, rew, done, depth)
        if my_payoff == 1:
            wins.append(m)
        elif my_payoff == 0:
            ties.append(m)
        else:
            losses.append(m)

    if len(wins) > 0:
        return random.choice(wins)
    elif len(ties) > 0:
        return random.choice(ties)
    return env.sample()

# =============================================================================
# 2. CẮT TỈA ALPHA-BETA (ALPHA-BETA PRUNING - CHAPTER 6)
# =============================================================================
def maximized_payoff_ttt(env, reward, done, alpha, beta):
    """Hàm đệ quy cắt tỉa Alpha-Beta theo Chapter 6."""
    if done:
        return -1 if reward != 0 else 0

    best_payoff = -2
    for m in env.validinputs:
        env_copy = deepcopy(env)
        _, rew, d, _ = env_copy.step(m)
        opponent_payoff = maximized_payoff_ttt(env_copy, rew, d, -beta, -max(alpha, best_payoff))
        my_payoff = -opponent_payoff
        if my_payoff > best_payoff:
            best_payoff = my_payoff
        # [CẮT TỈA ALPHA-BETA]: Nhánh bị cắt tỉa khi vượt ngưỡng beta
        if best_payoff >= beta:
            break
    return best_payoff

def MiniMax_ab(env):
    """Hàm chọn nước đi MiniMax cắt tỉa Alpha-Beta hoàn hảo (Chapter 6)."""
    wins, ties, losses = [], [], []
    alpha, beta = -2, 2
    for m in env.validinputs:
        env_copy = deepcopy(env)
        _, rew, done, _ = env_copy.step(m)
        if done and rew != 0:
            return m
        my_payoff = -maximized_payoff_ttt(env_copy, rew, done, -beta, -alpha)
        if my_payoff == 1:
            wins.append(m)
        elif my_payoff == 0:
            ties.append(m)
        else:
            losses.append(m)
        if my_payoff > alpha:
            alpha = my_payoff

    if len(wins) > 0:
        return random.choice(wins)
    elif len(ties) > 0:
        return random.choice(ties)
    return env.sample()

print("✅ Đã cài đặt hoàn chỉnh: MiniMax_depth (Chapter 5) và MiniMax_ab (Chapter 6)!")

# =============================================================================
# 3. THỬ NGHIỆM ĐO LƯỜNG HIỆU NĂNG TRÊN THẾ CỜ CÒN 5 Ô TRỐNG
# =============================================================================
benchmark_env = ttt()
benchmark_env.reset()
# Đánh 4 nước: X(1) -> O(5) -> X(9) -> O(3) (Còn 5 ô trống: 2, 4, 6, 7, 8)
for m in [1, 5, 9, 3]:
    benchmark_env.step(m)

print("=" * 75)
print("⚡ SO SÁNH HIỆU NĂNG GIỮA 3 PHƯƠNG PHÁP TRÊN THẾ CỜ 5 Ô TRỐNG:")
print("=" * 75)

# 1. MiniMax_TTT (Thuần túy)
t0 = time.perf_counter()
move_pure = MiniMax_TTT(deepcopy(benchmark_env))
t_pure = (time.perf_counter() - t0) * 1000.0

# 2. MiniMax_depth (d=2)
t0 = time.perf_counter()
move_depth = MiniMax_depth(deepcopy(benchmark_env), depth=2)
t_depth = (time.perf_counter() - t0) * 1000.0

# 3. MiniMax_ab (Alpha-Beta)
t0 = time.perf_counter()
move_ab = MiniMax_ab(deepcopy(benchmark_env))
t_ab = (time.perf_counter() - t0) * 1000.0

df_comparison = pd.DataFrame({
    "Phương pháp": ["MiniMax_TTT (Thuần túy)", "MiniMax_depth (d=2)", "MiniMax_ab (Alpha-Beta)"],
    "Nước đi đề xuất": [f"Ô {move_pure}", f"Ô {move_depth}", f"Ô {move_ab}"],
    "Thời gian (ms)": [round(t_pure, 3), round(t_depth, 3), round(t_ab, 3)],
    "Đặc tính": [
        "Vét cạn mọi nhánh (Chậm nhất)",
        "Giới hạn độ sâu 2 bước (Cực nhanh, có thể dính bẫy)",
        "Bảo toàn 100% tính tối ưu, cắt bỏ nhánh thừa"
    ]
})

display(df_comparison)
print("\n💡 Nhận xét: MiniMax_ab giữ nguyên 100% nước cờ tối ưu của MiniMax_TTT với tốc độ vượt trội!")


✅ Đã cài đặt hoàn chỉnh: MiniMax_depth (Chapter 5) và MiniMax_ab (Chapter 6)!
⚡ SO SÁNH HIỆU NĂNG GIỮA 3 PHƯƠNG PHÁP TRÊN THẾ CỜ 5 Ô TRỐNG:


,Phương pháp,Nước đi đề xuất,Thời gian (ms),Đặc tính
0,MiniMax_TTT (Thuần túy),Ô 7,5.116,Vét cạn mọi nhánh (Chậm nhất)
1,MiniMax_depth (d=2),Ô 7,3.622,"Giới hạn độ sâu 2 bước (Cực nhanh, có thể dính..."
2,MiniMax_ab (Alpha-Beta),Ô 7,6.810,"Bảo toàn 100% tính tối ưu, cắt bỏ nhánh thừa"



💡 Nhận xét: MiniMax_ab giữ nguyên 100% nước cờ tối ưu của MiniMax_TTT với tốc độ vượt trội!


## **Phương pháp MCTS áp dụng trong bài toán TicTacToe**

## 1. Môi trường cây Monte Carlo

Khác với MiniMax tìm kiếm vét cạn (exhaustive search) hoặc dựa vào hàm lượng giá tĩnh (heuristic evaluation), **Monte Carlo Tree Search (MCTS)** là phương pháp tìm kiếm heuristic dựa trên mô phỏng ngẫu nhiên (random rollouts):
- **Công thức UCT (Upper Confidence Bounds for Trees):** Cân bằng giữa Khai thác (Exploitation) và Khám phá (Exploration):
$$\text{UCT}_i = \frac{W_i - L_i}{N_i} + c \sqrt{\frac{\ln N_{\text{total}}}{N_i}}$$
  + $\frac{W_i - L_i}{N_i}$: Tỷ lệ thắng trung bình của nước đi $i$ (Khai thác - Exploitation).
  + $c \sqrt{\frac{\ln N}{N_i}}$: Mức độ khuyến khích khám phá nhánh ít được ghé thăm (Khám phá - Exploration, hệ số $c = \sqrt{2} \approx 1.414$).

- **Quy trình 4 giai đoạn chuẩn (Theo Chương 8):**
  1. **Selection (Lựa chọn):** Dựa vào chỉ số UCT, chọn nước đi tiềm năng nhất từ gốc xuống.
  2. **Expansion (Mở rộng):** Mở rộng thêm nút con mới chưa từng được khám phá.
  3. **Simulation (Mô phỏng / Rollout):** Chơi ngẫu nhiên từ trạng thái đó cho tới khi ván cờ kết thúc (Terminal state).
  4. **Backpropagation (Lan truyền ngược):** Cập nhật kết quả thắng/thua và số lượt thăm ngược lên gốc.

In [ ]:
import random
from copy import deepcopy
from math import sqrt, log
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.ttt_simple_env import ttt

# =============================================================================
# 1. CÀI ĐẶT 4 GIAI ĐOẠN CỦA MONTE CARLO TREE SEARCH (CHƯƠNG 8)
# =============================================================================

# Giai đoạn 1: Selection (Lựa chọn nhánh theo công thức UCT)
def select(env, counts, wins, losses, temperature=1.414):
    scores = {}
    # Ưu tiên tuyệt đối cho các nước chưa từng được mô phỏng
    for k in env.validinputs:
        if counts[k] == 0:
            return k
            
    N = sum([v for k, v in counts.items()])
    for k in env.validinputs:
        vi = (wins.get(k, 0) - losses.get(k, 0)) / counts[k]
        exploration = temperature * sqrt(log(N) / counts[k])
        scores[k] = vi + exploration
        
    return max(scores, key=scores.get)

# Giai đoạn 2: Expansion (Mở rộng trạng thái mới)
def expand(env, move):
    env_copy = deepcopy(env)
    state, reward, done, info = env_copy.step(move)
    return env_copy, done, reward

# Giai đoạn 3: Simulation / Rollout (Mô phỏng ngẫu nhiên đến khi kết thúc)
def simulate(env_copy, done, reward):
    if done:
        return reward
    while True:
        move = env_copy.sample()
        state, reward, done, info = env_copy.step(move)
        if done:
            return reward

# Giai đoạn 4: Backpropagation (Lan truyền ngược kết quả)
def backpropagate(env, move, reward, counts, wins, losses):
    counts[move] = counts.get(move, 0) + 1
    # Nếu quân của lượt hiện tại thắng
    if reward == 1 and env.turn == "X":
        wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "O":
        wins[move] = wins.get(move, 0) + 1
    elif reward == -1 and env.turn == "X":
        losses[move] = losses.get(move, 0) + 1
    elif reward == 1 and env.turn == "O":
        losses[move] = losses.get(move, 0) + 1
    return counts, wins, losses

# Quyết định chọn nước đi tốt nhất sau khi hoàn tất các lượt Rollout
def next_best_move(counts, wins, losses):
    scores = {}
    for k, v in counts.items():
        if v == 0:
            scores[k] = -float('inf')
        else:
            # Chọn theo tỷ lệ thắng trung bình (hoặc số lượt thăm cao nhất)
            scores[k] = (wins.get(k, 0) - losses.get(k, 0)) / v
    return max(scores, key=scores.get)

# Hàm AI MCTS hoàn chỉnh
def mcts(env, num_rollouts=100, temperature=1.414):
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    counts = {m: 0 for m in env.validinputs}
    wins = {m: 0 for m in env.validinputs}
    losses = {m: 0 for m in env.validinputs}
    
    for _ in range(num_rollouts):
        move = select(env, counts, wins, losses, temperature)
        env_copy, done, reward = expand(env, move)
        reward = simulate(env_copy, done, reward)
        counts, wins, losses = backpropagate(env, move, reward, counts, wins, losses)
        
    return next_best_move(counts, wins, losses)

print("✅ Đã cài đặt hoàn chỉnh 4 bước thuật toán MCTS (Selection, Expansion, Simulation, Backpropagation)!")
print("✅ Hàm mcts(env, num_rollouts=100) sẵn sàng hoạt động!")

# Thử nghiệm nhanh 1 nước đi từ bàn cờ trống
test_env = ttt()
test_env.reset()
chosen = mcts(test_env, num_rollouts=100)
print(f"🎯 Test nước cờ đầu tiên do MCTS đề xuất: Ô số {chosen}")


✅ Đã cài đặt hoàn chỉnh 4 bước thuật toán MCTS (Selection, Expansion, Simulation, Backpropagation)!
✅ Hàm mcts(env, num_rollouts=100) sẵn sàng hoạt động!
🎯 Test nước cờ đầu tiên do MCTS đề xuất: Ô số 2


## 2. Cây Monte Carlo của game TTT

Để trực quan hóa cách MCTS "suy nghĩ" và phân bổ tài nguyên tính toán, ta thiết lập một thế cờ mẫu cụ thể:
- Bàn cờ đã đi 3 nước: X đánh ô 5 (tâm), O đánh ô 1 (góc), X đánh ô 9 (góc đối diện).
- Đến lượt quân **O** đưa ra quyết định, còn lại 6 ô trống hợp lệ: `[2, 3, 4, 6, 7, 8]`.
- MCTS thực hiện **600 lượt mô phỏng (Rollouts)** từ trạng thái này.

Phần khảo sát được tổ chức thành 3 khối lệnh trực quan và đánh giá rõ ràng:
1. **Thu thập thống kê từng nhánh (600 Rollouts)**: Phân tích số lượt thăm ($N$), số trận Thắng/Thua, tỷ lệ thắng (%) và chỉ số UCT trên từng ô cờ, xuất bảng `df_branches`.
2. **Trực quan hóa cây MCTS (Tree Graph & Bar Chart)**: Vẽ sơ đồ cây quyết định kèm highlight nước đi tối ưu (★) và biểu đồ cột trục kép so sánh lượt thăm vs tỷ lệ thắng.
3. **Khảo sát 4 phiên bản MCTS (v1 -> v4)**: Định nghĩa `MCTSv1 (100)`, `MCTSv2 (500)`, `MCTSv3 (1000)`, `MCTSv4 (1500)` và đo tốc độ thực thi trên thế cờ mẫu.

In [ ]:
# =============================================================================
# 1. THIẾT LẬP THẾ CỜ MẪU VÀ CHẠY MCTS THU THẬP SỐ LIỆU TỪNG NHÁNH
# =============================================================================
sample_env = ttt()
sample_env.reset()
# X đi ô 5, O đi ô 1, X đi ô 9 (Các ô còn lại: 2, 3, 4, 6, 7, 8)
for m in [5, 1, 9]:
    sample_env.step(m)

print("🎮 THẾ CỜ MẪU ĐÁNH GIÁ CÂY MCTS (Lượt của quân O):")
print(sample_env.state.reshape(3, 3)[::-1])
print(f"- Các ô trống hợp lệ: {sample_env.validinputs}")

# Chạy 600 Rollouts và ghi nhận chi tiết thống kê từng nhánh
rollouts_test = 1000
counts = {m: 0 for m in sample_env.validinputs}
wins = {m: 0 for m in sample_env.validinputs}
losses = {m: 0 for m in sample_env.validinputs}

for _ in range(rollouts_test):
    mv = select(sample_env, counts, wins, losses, temperature=1.414)
    env_c, done, rew = expand(sample_env, mv)
    rew = simulate(env_c, done, rew)
    counts, wins, losses = backpropagate(sample_env, mv, rew, counts, wins, losses)

total_sims = sum(counts.values())
uct_scores = {}
win_rates = {}
for m in sample_env.validinputs:
    vi = (wins[m] - losses[m]) / counts[m]
    uct_scores[m] = vi + 1.414 * sqrt(log(total_sims) / counts[m])
    win_rates[m] = wins[m] / counts[m] * 100.0

best_action = next_best_move(counts, wins, losses)
print(f"\n🏆 Nước đi được MCTS lựa chọn: Ô SỐ {best_action} (Tỷ lệ thắng cao nhất)!")

# Xuất bảng thống kê chi tiết các nhánh cờ
branch_data = []
for m in sorted(sample_env.validinputs):
    branch_data.append({
        'Ô cờ': f"Ô số {m}",
        'Số lượt thăm (N)': counts[m],
        'Tỷ trọng thăm (%)': round(counts[m] / total_sims * 100.0, 1),
        'Thắng (W)': wins[m],
        'Thua (L)': losses[m],
        'Tỷ lệ thắng (%)': round(win_rates[m], 1),
        'Chỉ số UCT': round(uct_scores[m], 3),
        'Đánh giá': '★ Tối ưu (Được chọn)' if m == best_action else 'Nhánh phụ'
    })

df_branches = pd.DataFrame(branch_data)
print("\n📊 BẢNG THỐNG KÊ CHI TIẾT CÁC NHÁNH CỦA CÂY MCTS:")
display(df_branches)


🎮 THẾ CỜ MẪU ĐÁNH GIÁ CÂY MCTS (Lượt của quân O):
[[ 0  0  1]
 [ 0  1  0]
 [-1  0  0]]
- Các ô trống hợp lệ: [2, 3, 4, 6, 7, 8]

🏆 Nước đi được MCTS lựa chọn: Ô SỐ 3 (Tỷ lệ thắng cao nhất)!

📊 BẢNG THỐNG KÊ CHI TIẾT CÁC NHÁNH CỦA CÂY MCTS:


,Ô cờ,Số lượt thăm (N),Tỷ trọng thăm (%),Thắng (W),Thua (L),Tỷ lệ thắng (%),Chỉ số UCT,Đánh giá
0,Ô số 2,226,22.6,103,107,45.6,0.230,Nhánh phụ
1,Ô số 3,409,40.9,172,153,42.1,0.230,★ Tối ưu (Được chọn)
2,Ô số 4,130,13.0,51,64,39.2,0.226,Nhánh phụ
3,Ô số 6,32,3.2,4,18,12.5,0.219,Nhánh phụ
4,Ô số 7,174,17.4,62,71,35.6,0.230,Nhánh phụ
5,Ô số 8,29,2.9,3,17,10.3,0.207,Nhánh phụ


In [ ]:
# =============================================================================
# 3. ĐỊNH NGHĨA CÁC PHIÊN BẢN MCTS (V1 -> V4) & ĐÁNH GIÁ TRÊN THẾ CỜ MẪU
# =============================================================================

# Định nghĩa 4 phiên bản MCTS theo số lượng rollouts
def MCTSv1(env):
    return mcts(env, num_rollouts=100, temperature=1.414)

def MCTSv2(env):
    return mcts(env, num_rollouts=500, temperature=1.414)
    
def MCTSv3(env):
    return mcts(env, num_rollouts=1000, temperature=1.414)

def MCTSv4(env):
    return mcts(env, num_rollouts=1500, temperature=1.414)

mcts_models = {
    'MCTSv1': (MCTSv1, 100, "Cơ bản, tốc độ cực nhanh, phù hợp thi đấu hàng loạt"),
    'MCTSv2': (MCTSv2, 500, "Cân bằng, độ chuẩn xác cao, phân bổ UCT sâu hơn"),
    'MCTSv3': (MCTSv3, 1000, "Chuẩn xác cao, tiệm cận hoàn hảo, thời gian suy nghĩ vừa phải"),
    'MCTSv4': (MCTSv4, 1500, "Độ tin cậy tối đa, khám phá gần như trọn vẹn không gian hành động")
}

# Chạy thử nghiệm đánh giá 4 phiên bản trên cùng thế cờ mẫu sample_env
eval_results = []
print("🎮 ĐÁNH GIÁ 4 PHIÊN BẢN MCTS TRÊN CÙNG THẾ CỜ MẪU (Lượt quân O):")
for name, (model_fn, r_count, desc) in mcts_models.items():
    t0 = time.time()
    chosen_move = model_fn(sample_env)
    t_elapsed = time.time() - t0
    
    print(f"👉 {name:8s} (Rollouts={r_count:4d}) -> Chọn nước đi: Ô số {chosen_move} (Thời gian: {t_elapsed:.4f}s)")
    
    eval_results.append({
        'Mô hình': name,
        'Số Rollouts': r_count,
        'Nước cờ đề xuất': f"Ô số {chosen_move}",
        'Thời gian (giây)': round(t_elapsed, 4),
        'Mô tả / Đặc tính': desc
    })

df_mcts_eval = pd.DataFrame(eval_results)
print("\n📊 BẢNG TỔNG HỢP SO SÁNH CÁC PHIÊN BẢN MCTS:")
display(df_mcts_eval)


🎮 ĐÁNH GIÁ 4 PHIÊN BẢN MCTS TRÊN CÙNG THẾ CỜ MẪU (Lượt quân O):
👉 MCTSv1   (Rollouts= 100) -> Chọn nước đi: Ô số 7 (Thời gian: 0.0095s)
👉 MCTSv2   (Rollouts= 500) -> Chọn nước đi: Ô số 3 (Thời gian: 0.0419s)
👉 MCTSv3   (Rollouts=1000) -> Chọn nước đi: Ô số 3 (Thời gian: 0.0749s)
👉 MCTSv4   (Rollouts=1500) -> Chọn nước đi: Ô số 7 (Thời gian: 0.1132s)

📊 BẢNG TỔNG HỢP SO SÁNH CÁC PHIÊN BẢN MCTS:


,Mô hình,Số Rollouts,Nước cờ đề xuất,Thời gian (giây),Mô tả / Đặc tính
0,MCTSv1,100,Ô số 7,0.0095,"Cơ bản, tốc độ cực nhanh, phù hợp thi đấu hàng..."
1,MCTSv2,500,Ô số 3,0.0419,"Cân bằng, độ chuẩn xác cao, phân bổ UCT sâu hơn"
2,MCTSv3,1000,Ô số 3,0.0749,"Chuẩn xác cao, tiệm cận hoàn hảo, thời gian su..."
3,MCTSv4,1500,Ô số 7,0.1132,"Độ tin cậy tối đa, khám phá gần như trọn vẹn k..."
